# PocketGPT — fine-tune on Colab (C3)

Runtime → Change runtime type → **T4 GPU**. Then *Runtime → Run all* (~40–60 min).

1. Install the tested library versions.
2. Get the repository (the code and the frozen dataset in `data/pocket_chat/`).
3. QLoRA fine-tune `Qwen/Qwen2.5-1.5B-Instruct` with `training/train.py` (the same script that was smoke-tested on a Mac).
4. Compare base vs fine-tuned on a few test questions.
5. Merge → GGUF (Q4_K_M, ~1 GB) with llama.cpp, and download it.

On the Mac: put the file in `pocket-gpt/models/` and start `llama-server` with it (see README).

In [ ]:
# 1. Libraries — the versions training/train.py was tested with (Mac smoke test), plus bitsandbytes for 4-bit.
!pip -q install "transformers==5.22.1" "peft==0.21.2" "trl==1.17.1" "datasets==5.1.0" "accelerate==1.15.0" bitsandbytes
import torch
print(torch.__version__, torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU — switch the runtime to T4')

## 2. The repository

Push `pocket-gpt` to GitHub first (see README). A private repository needs a token: *Secrets* (🔑 on the left) → `GITHUB_TOKEN`.
If you prefer, upload a zip of the repository instead and unzip it to `/content/pocket-gpt`.

In [ ]:
import os
REPO = 'yakhyo018/Pocket-GPT'   # ← change if the repository has another name
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    token = None
url = f'https://{token}@github.com/{REPO}.git' if token else f'https://github.com/{REPO}.git'
if not os.path.exists('/content/pocket-gpt'):
    !git clone -q {url} /content/pocket-gpt
%cd /content/pocket-gpt
!wc -l data/pocket_chat/*.jsonl

## 3. Fine-tune (QLoRA)

4-bit base + LoRA r=16 on every linear layer; loss on the assistant's reply only; 1 epoch over 3,600 chats (EN/RU/UZ, 1,200 each); validation loss every 50 steps.
`--merge` also saves base + adapter merged in fp16 for the GGUF step.

In [ ]:
!python training/train.py --model Qwen/Qwen2.5-1.5B-Instruct --qlora --epochs 1 --batch 4 --accum 4 --max-length 1024 --merge

In [ ]:
import json
report = json.load(open('outputs/pocket-lora/training.json'))
print({k: v for k, v in report.items() if k != 'history'})
for h in report['history']:
    if 'eval_loss' in h: print('step', h['step'], 'eval_loss', round(h['eval_loss'], 3))

## 4. Base vs fine-tuned — a quick look (the full evaluation runs on the Mac: `training/evaluate.py`)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

SYSTEM = json.loads(open('data/pocket_chat/test.jsonl').readline())['messages'][0]['content']
QUESTIONS = [
    "Salom! O'zbekistonning poytaxti qayer va u yerda nimalarni ko'rish mumkin?",
    'Привет! Посоветуй, как быстрее выучить новый язык.',
    'Explain in two sentences what a LoRA adapter is.',
]
tok = AutoTokenizer.from_pretrained('Qwen/Qwen2.5-1.5B-Instruct')

def answers(path):
    model = AutoModelForCausalLM.from_pretrained(path, dtype=torch.float16, device_map='cuda')
    out = []
    for q in QUESTIONS:
        ids = tok.apply_chat_template([{'role': 'system', 'content': SYSTEM}, {'role': 'user', 'content': q}],
                                      add_generation_prompt=True, return_tensors='pt', return_dict=True).to('cuda')
        gen = model.generate(**ids, max_new_tokens=160, do_sample=False, repetition_penalty=1.1)
        out.append(tok.decode(gen[0][ids['input_ids'].shape[1]:], skip_special_tokens=True).strip())
    del model; torch.cuda.empty_cache()
    return out

base, tuned = answers('Qwen/Qwen2.5-1.5B-Instruct'), answers('outputs/pocket-merged')
for q, b, t in zip(QUESTIONS, base, tuned):
    print('Q:', q, '\n  BASE :', b[:400], '\n  TUNED:', t[:400], '\n')

## 5. GGUF for the Mac (llama.cpp)

`convert_hf_to_gguf.py` writes an fp16 GGUF; `llama-quantize` makes the ~1 GB Q4_K_M file that `llama-server` runs.

In [ ]:
!git clone -q --depth 1 https://github.com/ggml-org/llama.cpp /content/llama.cpp
!pip -q install -e /content/llama.cpp/gguf-py
!python /content/llama.cpp/convert_hf_to_gguf.py outputs/pocket-merged --outtype f16 --outfile outputs/pocketgpt-f16.gguf
!cmake -S /content/llama.cpp -B /content/llama.cpp/build -DGGML_CUDA=OFF -DLLAMA_CURL=OFF > /dev/null
!cmake --build /content/llama.cpp/build --target llama-quantize -j 4 > /dev/null
!/content/llama.cpp/build/bin/llama-quantize outputs/pocketgpt-f16.gguf outputs/pocketgpt-1.5b-q4_k_m.gguf Q4_K_M
!ls -lh outputs/*.gguf

In [ ]:
# Download: the GGUF (~1 GB) and the small adapter + training report.
!cd outputs && zip -qr pocket-lora.zip pocket-lora
from google.colab import files
files.download('outputs/pocketgpt-1.5b-q4_k_m.gguf')
files.download('outputs/pocket-lora.zip')